In [2]:
# Sauvegarde du fichier nettoyé pour Streamlit
df_clean.to_csv('../data/processed/taux_chomage_regions_clean.csv', index=False)

print("Fichier sauvegardé avec succès dans data/processed/ !")

Fichier sauvegardé avec succès dans data/processed/ !


In [3]:
import openpyxl
import pandas as pd

In [4]:
import os
print(os.getcwd())

C:\Users\Lenovo\regional-unemployment-dashboard\notebooks


In [5]:
import openpyxl
import pandas as pd

wb = openpyxl.load_workbook('../data/raw/taux_chomage_sexe_region_2015-2025.xlsx', data_only=True)
ws = wb['Données']

years = [c.value for c in ws[1]][3:]
years = [y for y in years if y is not None]

rows = []
for row in ws.iter_rows(min_row=3, max_row=ws.max_row):
    region = row[1].value
    if region is None:
        continue
    for i, year in enumerate(years):
        val = row[3+i].value
        if val is not None:
            rows.append({'annee': int(year), 'region_raw': region, 'taux_chomage': float(val)})

df_clean = pd.DataFrame(rows)
df_clean.head()

,annee,region_raw,taux_chomage
0,2025,Tanger-Tétouan-Al Hoceïma,9.4
1,2024,Tanger-Tétouan-Al Hoceïma,10.2
2,2023,Tanger-Tétouan-Al Hoceïma,10.1
3,2022,Tanger-Tétouan-Al Hoceïma,9.7
4,2021,Tanger-Tétouan-Al Hoceïma,10.8


In [6]:
df_clean['region'] = (
    df_clean['region_raw']
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.replace(' -', '-', regex=False)
    .str.replace('- ', '-', regex=False)
)
df_clean['region'] = df_clean['region'].str.replace('Casablanca-settat', 'Casablanca-Settat', regex=False)
df_clean['region'] = df_clean['region'].str.replace('Marrakech-safi', 'Marrakech-Safi', regex=False)

df_clean['is_national'] = df_clean['region'].isin(['Ensemble', 'National'])
df_clean.loc[df_clean['is_national'], 'region'] = 'National'

df_clean = df_clean[['annee', 'region', 'taux_chomage', 'is_national']]
df_clean = df_clean.sort_values(['region', 'annee']).reset_index(drop=True)

print(df_clean.shape)
df_clean.head()

(121, 4)


,annee,region,taux_chomage,is_national
0,2015,Béni Mellal-Khénifra,6.6,False
1,2016,Béni Mellal-Khénifra,7.1,False
2,2017,Béni Mellal-Khénifra,6.1,False
3,2018,Béni Mellal-Khénifra,5.8,False
4,2019,Béni Mellal-Khénifra,5.4,False


In [7]:
print("Valeurs manquantes:\n", df_clean.isnull().sum())
print("\nDoublons (annee, region):", df_clean.duplicated(subset=['annee','region']).sum())
print("\nRégions présentes:", sorted(df_clean['region'].unique()))
print("\nAnnées présentes:", sorted(df_clean['annee'].unique()))
print("\nStatistiques du taux de chômage:")
print(df_clean['taux_chomage'].describe())

Valeurs manquantes:
 annee           0
region          0
taux_chomage    0
is_national     0
dtype: int64

Doublons (annee, region): 0

Régions présentes: ['Béni Mellal-Khénifra', 'Casablanca-Settat', 'Drâa-Tafilalet', 'Fès-Meknès', 'Marrakech-Safi', 'National', 'Oriental', 'Rabat-Salé-Kénitra', 'Régions du Sud', 'Souss-Massa', 'Tanger-Tétouan-Al Hoceïma']

Années présentes: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Statistiques du taux de chômage:
count    121.000000
mean      11.615702
std        3.988734
min        5.300000
25%        9.000000
50%       10.800000
75%       13.500000
max       22.800000
Name: taux_chomage, dtype: float64


In [8]:
def get_ranking(df, annee):
    """Classement des régions (hors National) pour une année donnée, du taux le plus haut au plus bas."""
    d = df[(df['annee'] == annee) & (~df['is_national'])].copy()
    d = d.sort_values('taux_chomage', ascending=False).reset_index(drop=True)
    d['rang'] = d.index + 1
    return d[['rang', 'region', 'taux_chomage']]


def get_national_avg(df, annee):
    """Taux de chômage national pour une année donnée."""
    d = df[(df['annee'] == annee) & (df['is_national'])]
    return float(d['taux_chomage'].iloc[0]) if not d.empty else None


def get_summary_stats(df, annee):
    """Moyenne nationale, région max, région min pour une année donnée."""
    ranking = get_ranking(df, annee)
    return {
        'moyenne_nationale': get_national_avg(df, annee),
        'region_max': ranking.iloc[0]['region'],
        'valeur_max': ranking.iloc[0]['taux_chomage'],
        'region_min': ranking.iloc[-1]['region'],
        'valeur_min': ranking.iloc[-1]['taux_chomage'],
    }


def get_evolution(df, region):
    """Série temporelle du taux de chômage pour une région (ou 'National')."""
    if region == 'National':
        d = df[df['is_national']]
    else:
        d = df[df['region'] == region]
    return d[['annee', 'taux_chomage']].sort_values('annee').reset_index(drop=True)


def compare_regions(df, region1, region2):
    """Fusionne les évolutions de 2 régions pour un graphique comparatif."""
    e1 = get_evolution(df, region1).rename(columns={'taux_chomage': region1})
    e2 = get_evolution(df, region2).rename(columns={'taux_chomage': region2})
    return e1.merge(e2, on='annee', how='outer').sort_values('annee').reset_index(drop=True)


def get_region_rank_over_time(df, region):
    """Rang de la région pour chaque année disponible (utile pour la page Focus)."""
    result = []
    for annee in sorted(df['annee'].unique()):
        ranking = get_ranking(df, annee)
        match = ranking[ranking['region'] == region]
        if not match.empty:
            result.append({'annee': annee, 'rang': int(match.iloc[0]['rang']), 'taux_chomage': match.iloc[0]['taux_chomage']})
    return pd.DataFrame(result)

In [9]:
print(get_ranking(df_clean, 2023))
print()
print(get_summary_stats(df_clean, 2023))
print()
print(get_evolution(df_clean, 'Tanger-Tétouan-Al Hoceïma'))
print()
print(get_region_rank_over_time(df_clean, 'Tanger-Tétouan-Al Hoceïma'))

   rang                     region  taux_chomage
0     1             Régions du Sud          20.3
1     2                   Oriental          19.6
2     3          Casablanca-Settat          15.0
3     4                 Fès-Meknès          14.2
4     5                Souss-Massa          13.5
5     6       Béni Mellal-Khénifra          12.8
6     7             Drâa-Tafilalet          11.9
7     8         Rabat-Salé-Kénitra          11.6
8     9  Tanger-Tétouan-Al Hoceïma          10.1
9    10             Marrakech-Safi           7.7

{'moyenne_nationale': 13.0, 'region_max': 'Régions du Sud', 'valeur_max': np.float64(20.3), 'region_min': 'Marrakech-Safi', 'valeur_min': np.float64(7.7)}

    annee  taux_chomage
0    2015          10.7
1    2016           9.6
2    2017           8.2
3    2018           7.5
4    2019           8.6
5    2020          10.4
6    2021          10.8
7    2022           9.7
8    2023          10.1
9    2024          10.2
10   2025           9.4

    annee  rang

In [10]:
def filter_data(df, year, sex):
    return df[
        (df["annee"] == year) &
        (df["sexe"] == sex)
    ].copy()


def rank_regions(df):
    df = df.sort_values("taux_chomage", ascending=False).copy()
    df["rang"] = range(1, len(df) + 1)
    return df


def get_summary(df, focus_region="Tanger-Tétouan-Al Hoceïma"):
    average = df["taux_chomage"].mean()
    max_row = df.loc[df["taux_chomage"].idxmax()]
    min_row = df.loc[df["taux_chomage"].idxmin()]
    focus = df[df["region"] == focus_region]

    return average, max_row, min_row, focus

In [11]:
import openpyxl
import pandas as pd

# 1. Extraction des données
wb = openpyxl.load_workbook('../data/raw/taux_chomage_sexe_region_2015-2025.xlsx', data_only=True)
ws = wb['Données']

years = [c.value for c in ws[1]][3:]
years = [y for y in years if y is not None]

rows = []
for row in ws.iter_rows(min_row=3, max_row=ws.max_row):
    region = row[1].value
    if region is None:
        continue
    for i, year in enumerate(years):
        val = row[3+i].value
        if val is not None:
            rows.append({'annee': int(year), 'region_raw': region, 'taux_chomage': float(val)})

df_clean = pd.DataFrame(rows)

# 2. Nettoyage des noms de régions
df_clean['region'] = (
    df_clean['region_raw']
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.replace(' -', '-', regex=False)
    .str.replace('- ', '-', regex=False)
)
df_clean['region'] = df_clean['region'].str.replace('Casablanca-settat', 'Casablanca-Settat', regex=False)
df_clean['region'] = df_clean['region'].str.replace('Marrakech-safi', 'Marrakech-Safi', regex=False)

# 3. CRÉATION DE LA COLONNE MANQUANTE 'is_national'
df_clean['is_national'] = df_clean['region'].isin(['Ensemble', 'National'])
df_clean.loc[df_clean['is_national'], 'region'] = 'National'

# 4. Finalisation avec les colonnes exactes attendues par Streamlit
df_clean = df_clean[['annee', 'region', 'taux_chomage', 'is_national']]
df_clean = df_clean.sort_values(['region', 'annee']).reset_index(drop=True)

# 5. Sauvegarde
output_path = '../data/processed/taux_chomage_regions_clean.csv'
df_clean.to_csv(output_path, index=False, encoding='utf-8')

print(f"✅ SUCCÈS : Le fichier a retrouvé sa bonne structure et est sauvegardé dans {output_path}")

✅ SUCCÈS : Le fichier a retrouvé sa bonne structure et est sauvegardé dans ../data/processed/taux_chomage_regions_clean.csv


In [12]:
import openpyxl
import pandas as pd

# 1. Chargement et extraction
wb = openpyxl.load_workbook('../data/raw/taux_chomage_sexe_region_2015-2025.xlsx', data_only=True)
ws = wb['Données']

years = [c.value for c in ws[1]][3:]
years = [y for y in years if y is not None]

rows = []
for row in ws.iter_rows(min_row=3, max_row=ws.max_row):
    region = row[1].value
    if region is None:
        continue
    for i, year in enumerate(years):
        val = row[3+i].value
        if val is not None:
            rows.append({'annee': int(year), 'region_raw': region, 'taux_chomage': float(val)})

df = pd.DataFrame(rows)

# 2. Nettoyage et formatage
df['region'] = df['region_raw'].str.strip().str.replace(r'\s+', ' ', regex=True).str.replace(' -', '-').str.replace('- ', '-')
df['region'] = df['region'].str.replace('Casablanca-settat', 'Casablanca-Settat').str.replace('Marrakech-safi', 'Marrakech-Safi')

# 3. Création des colonnes spécifiques pour Streamlit
df['is_national'] = df['region'].isin(['Ensemble', 'National'])
df.loc[df['is_national'], 'region'] = 'National'

# 4. Sélection stricte et sauvegarde
df_clean = df[['annee', 'region', 'taux_chomage', 'is_national']].copy()
df_clean = df_clean.sort_values(['region', 'annee']).reset_index(drop=True)

df_clean.to_csv('../data/processed/taux_chomage_regions_clean.csv', index=False)
print("✅ Fichier généré avec succès ! Les colonnes actuelles sont :", list(df_clean.columns))

✅ Fichier généré avec succès ! Les colonnes actuelles sont : ['annee', 'region', 'taux_chomage', 'is_national']
